# Study C granular invariance analysis

Keep this notebook separate from the canonical `study_c_analysis.ipynb`.

Use it for:

- per-case recall decay slices
- persona/risk memory profiles
- contradiction exemplars
- summary-framing and turn-order robustness breakdowns

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

RUNTIME_ROOT = Path.cwd().resolve().parents[0]
CASE_DELTAS_PATH = RUNTIME_ROOT / "metric-results" / "invariance_smoke" / "qwen3-lmstudio" / "study_c_case_deltas.json"
CONTROLLABILITY_PATH = RUNTIME_ROOT / "metric-results" / "controllability" / "qwen3-lmstudio" / "study_c_archived_smoke.json"

rows = json.loads(CASE_DELTAS_PATH.read_text(encoding="utf-8")) if CASE_DELTAS_PATH.exists() else []
df = pd.DataFrame(rows)
control_payload = json.loads(CONTROLLABILITY_PATH.read_text(encoding="utf-8")) if CONTROLLABILITY_PATH.exists() else {}

print(f"case rows: {len(df)}")
display(df.head() if not df.empty else pd.DataFrame())

In [ ]:
if not df.empty:
    if "persona" not in df.columns and "metadata" in df.columns:
        df["persona"] = df["metadata"].apply(lambda value: value.get("persona_id") if isinstance(value, dict) else None)
    if "risk" not in df.columns and "strata" in df.columns:
        df["risk"] = df["strata"].apply(lambda value: value.get("risk") if isinstance(value, dict) else None)

    display(
        df.groupby(["metric", "persona"], as_index=False)
        .agg(mean_delta=("delta", "mean"), n=("id", "count"))
        .sort_values(["metric", "mean_delta"], ascending=[True, True])
        .head(40)
    )

    for metric_name in sorted(df["metric"].unique()):
        subset = df[df["metric"] == metric_name]
        plt.figure(figsize=(8, 4))
        subset.boxplot(column="delta", by="risk", grid=False)
        plt.suptitle("")
        plt.title(f"Study C delta by risk: {metric_name}")
        plt.ylabel("variant - base")
        plt.tight_layout()
        plt.show()

records = []
for variant in control_payload.get("variants", []):
    for metric_name, metric in variant.get("metrics", {}).items():
        records.append({
            "tag": variant.get("tag"),
            "metric": metric_name,
            "delta_c": metric.get("delta_c"),
            "ci_low": metric.get("ci_low"),
            "ci_high": metric.get("ci_high"),
        })
if records:
    display(pd.DataFrame(records))